# `lateralcauchy` — Experimentos del artículo

Interfaz única de los experimentos del paquete `lateralcauchy` (PINN para el
problema de Cauchy lateral del calor en un cilindro). Cada experimento es una
función de `lateralcauchy.experiments` cuyos **defaults son exactamente el spec**
(`docs/SIMULACIONES.md`); este notebook solo importa y llama.

| Función | Experimento | Sección del artículo |
|---|---|---|
| `ex1()` | Control de maquinaria (solución manufacturada) | 5.1 |
| `ex2()` | Frecuencia espacial (modo de Bessel m=0, n=1) | 5.2 |
| `ex3()` | Medio heterogéneo k(z) = 1 + z | 5.3 |
| `ex4()` | Robustez al ruido | 5.4 |
| `ex5()` | Mapa de degradación (**resultado central**) | 5.5 |
| `ex6()` | Sensibilidad al balance de pesos | 5.6 |
| `visualizacion()` | Visualización espacio-temporal | 5.7 |

Cada llamada imprime el diagnóstico `regimen()` al inicio, escribe `raw.csv`
(esquema §0.4), guarda modelos y emite las figuras del contrato; retorna un
`dict` con las rutas y un `resumen` (E0 media ± std por caso).

In [ ]:
# Instalación (Colab). En un clone local con el paquete instalado (pip install -e .)
# esta celda no hace nada.
try:
    import lateralcauchy  # noqa: F401
except ImportError:
    %pip install -q git+https://github.com/cmurillos/PINNs.git

In [ ]:
# Parámetros globales
SMOKE = True              # True: presupuesto reducido (minutos). False: presupuesto FULL del spec (horas/días).
OUTDIR = "results_colab"  # todas las salidas (csv, modelos, figuras) van bajo este directorio

# Para persistir en Google Drive, descomentar:
# from google.colab import drive
# drive.mount("/content/drive")
# OUTDIR = "/content/drive/MyDrive/lateralcauchy_results"

from IPython.display import Image, display
from lateralcauchy import experiments

def figuras(res):
    """Muestra inline los PNG hermanos de las figuras PDF retornadas."""
    for f in res["figs"]:
        display(Image(filename=f.replace(".pdf", ".png")))

In [ ]:
# Diagnóstico de atenuación (§1 del spec) sobre los dos medios + curva a(ω)
import numpy as np
from lateralcauchy.diagnostics import atenuacion, regimen

MEDIOS = {"constante (rc=k=1)": experiments.MEDIO_CONST,
          "k(z)=1+z (ex3)": (lambda z: np.ones_like(z), experiments.K_PERFIL_EX3)}
t = np.linspace(0.0, 1.0, 1001)
g = np.cos(5.94 * t)                       # armónico de la banda de ex2
for nombre, medio in MEDIOS.items():
    print(f"medio {nombre}:")
    print("   a(omega=5.94) =", round(atenuacion(5.94, medio, 1.0), 4))
    print("   regimen(eta=1e-2):", regimen((g, t[1] - t[0]), medio, 1.0, eta=1e-2))

figuras(experiments.aten_curvas(outdir=OUTDIR))

In [ ]:
# ex1 — control de maquinaria — criterio E0 ≲ 1e-2
res_ex1 = experiments.ex1(smoke=SMOKE, outdir=OUTDIR)
print(res_ex1['resumen'])
figuras(res_ex1)

In [ ]:
# ex2 — frecuencia espacial (Bessel), ω por bisección hasta a≈e⁻¹
res_ex2 = experiments.ex2(smoke=SMOKE, outdir=OUTDIR)
print(res_ex2['resumen'])
figuras(res_ex2)

In [ ]:
# ex3 — medio heterogéneo k(z)=1+z, misma a objetivo
res_ex3 = experiments.ex3(smoke=SMOKE, outdir=OUTDIR)
print(res_ex3['resumen'])
figuras(res_ex3)

In [ ]:
# ex4 — robustez al ruido η ∈ {0, 1e-3, 1e-2, 5e-2}
res_ex4 = experiments.ex4(smoke=SMOKE, outdir=OUTDIR)
print(res_ex4['resumen'])
figuras(res_ex4)

In [ ]:
# ex5 — mapa de degradación: colapso de E0 vs 1/a (RESULTADO CENTRAL)
res_ex5 = experiments.ex5(smoke=SMOKE, outdir=OUTDIR)
print(res_ex5['resumen'])
figuras(res_ex5)

In [ ]:
# ex6 — sensibilidad al balance (λ_g, λ_f)
res_ex6 = experiments.ex6(smoke=SMOKE, outdir=OUTDIR)
print(res_ex6['resumen'])
figuras(res_ex6)

In [ ]:
# Visualización 5.7 — carga el modelo guardado de ex3 (no reentrena)
res_vis = experiments.visualizacion(outdir=OUTDIR)
display(Image(filename=res_vis["figs"][0].replace(".pdf", ".png")))  # retícula 3x4
display(Image(filename=res_vis["figs"][1]))                          # GIF animado

## ⚠️ Presupuesto de cómputo

Con `SMOKE=False` cada corrida usa el presupuesto FULL del spec (15 000 Adam +
3 000 L-BFGS): **`ex5` son ~50–70 entrenamientos** y `ex4` otros ~48. Antes de
lanzarlos, estima el costo con UNA corrida de `ex2` completa
(`experiments.ex2(seeds=[0], smoke=False, outdir=OUTDIR)`) y multiplica por el
número de casos; en CPU puede ser de días — usa un runtime con GPU. El propio
`ex5` verifica el presupuesto con una corrida base antes de lanzar e imprime la
proyección.